In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

import os
import dotenv
API_KEY, HF_TOKEN = dotenv.dotenv_values("secret.env").values()

# Log in to Hugging Face
from huggingface_hub import snapshot_download, login
login(token=HF_TOKEN)

del HF_TOKEN

model_id = "Qwen/Qwen2.5-1.5B" 
local_folder = "./models/qwen2.5" # Change to your preferred path

with os.scandir("./models/qwen2.5") as it:
    print("Downloading Qwen2.5-1.5B...")
    if not any(it):
        download_path = snapshot_download(
            repo_id=model_id,
            local_dir=local_folder,
            ignore_patterns=["*.msgpack", "*.h5", "*.ot", "rust_model.ot", "original/*"],
            max_workers=4 # Downloads 4 files at a time to speed things up
        )

    print(f"Download complete! Model saved locally at: {download_path}")

# Options: "gpt2" (124M), "gpt2-medium" (355M), "gpt2-large" (774M), "gpt2-xl" (1.5B)
# model_id = "gpt2" 

# # Load the tokenizer
# tokenizer = AutoTokenizer.from_pretrained(model_id)

# # Load the model
# base_model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

/home/user/miniforge3/envs/capstone/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [ ]:
import os


[<DirEntry 'qwen2.5'>, <DirEntry 'gpt2'>]


In [ ]:
geometric_tokens = [
    # --- Control & Formatting Tokens ---
    "<GEO_START>", "<GEO_END>", "<DESCRIBE>", "<PAD>",
    
    # --- ISO 10303 Topology Tokens (The Graph) ---
    "<VERTEX_POINT>", 
    "<EDGE_CURVE>", "<ORIENTED_EDGE>", 
    "<EDGE_LOOP>", 
    "<FACE_BOUND>", "<FACE_OUTER_BOUND>", "<ADVANCED_FACE>", 
    "<CLOSED_SHELL>", "<OPEN_SHELL>", 
    "<MANIFOLD_SOLID_BREP>",
    
    # --- ISO 10303 Surface Geometry Tokens (The 2D Math) ---
    "<PLANE>", 
    "<CYLINDRICAL_SURFACE>", "<CONICAL_SURFACE>", 
    "<SPHERICAL_SURFACE>", "<TOROIDAL_SURFACE>", 
    "<SURFACE_OF_LINEAR_EXTRUSION>", "<SURFACE_OF_REVOLUTION>", 
    "<B_SPLINE_SURFACE_WITH_KNOTS>", "<RATIONAL_B_SPLINE_SURFACE>",
    
    # --- ISO 10303 Curve Geometry Tokens (The 1D Math) ---
    "<LINE>", "<CIRCLE>", "<ELLIPSE>", 
    "<PARABOLA>", "<HYPERBOLA>", 
    "<B_SPLINE_CURVE_WITH_KNOTS>", "<RATIONAL_B_SPLINE_CURVE>",
    
    # --- ISO 10303 Primitive Math Tokens ---
    "<CARTESIAN_POINT>", "<DIRECTION>", "<VECTOR>", "<AXIS2_PLACEMENT_3D>",

    # --- Macro compressed tokens ---
    "<CYLINDER_PRIMITIVE>", # A CYLINDRICAL_SURFACE node connected to exactly two CIRCLE edge nodes
    "<FILLET_CHAIN>", # Two or more B_SPLINE_SURFACE patches that share a common boundary edge and exhibit C^1 (tangential) continuity
    "<THROUGH_HOLE>", # A CYLINDRICAL_SURFACE where its two bounding CIRCLE edges are each connected to separate, distinct PLANE surfaces with opposing normal vectors.
    "<CHAMFER_EDGE>", #A CONICAL_SURFACE or narrow PLANE that bridges two orthogonal PLANE surfaces, bounded by parallel LINE or CIRCLE edges. 
    "<SPHERE_PRIMITIVE>", # A SPHERICAL_SURFACE bounded by a single CIRCLE edge (forming a hemisphere or ball joint) or a single vertex pole.
    "<PLANAR_PAD>" # A flat PLANE bounded by an EDGE_LOOP, where every edge in the loop connects perpendicularly to a surrounding "skirt" of PLANE or CYLINDRICAL_SURFACE nodes.
]

num_added_toks = tokenizer.add_special_tokens({'additional_special_tokens': geometric_tokens})
print(f"Added {num_added_toks} tokens. New vocab size: {len(tokenizer)}")

# Ensure there is a padding token
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<PAD>'})

In [ ]:
import networkx as nx
from networkx.algorithms import isomorphism

def compress_cylinder_macros(brep_graph):
    """
    Scans a B-Rep graph and compresses CYLINDRICAL_SURFACE + 2 CIRCLEs
    into a single <CYLINDER_PRIMITIVE> macro token.
    """
    # 1. Define the template subgraph we are looking for
    # Node types are stored in the 'entity_type' attribute
    cylinder_template = nx.DiGraph()
    cylinder_template.add_node("cyl", entity_type="CYLINDRICAL_SURFACE")
    cylinder_template.add_node("c1", entity_type="CIRCLE")
    cylinder_template.add_node("c2", entity_type="CIRCLE")
    
    # Define the expected topological edges (Cylinder bounds to Circles)
    cylinder_template.add_edge("cyl", "c1")
    cylinder_template.add_edge("cyl", "c2")

    # 2. Define the node matching function
    # This tells the matcher to only match nodes if their 'entity_type' is identical
    def node_match(n1, n2):
        return n1.get('entity_type') == n2.get('entity_type')

    # 3. Initialize the Graph Matcher
    matcher = isomorphism.DiGraphMatcher(brep_graph, cylinder_template, node_match=node_match)
    
    # Keep track of nodes to remove and add after iteration 
    # (modifying a graph while iterating over it causes errors)
    subgraphs_to_collapse = list(matcher.subgraph_isomorphisms_iter())
    macro_id_counter = 1

    

    for match in subgraphs_to_collapse:
        # 'match' is a dict mapping B-Rep node IDs to Template node IDs
        # Example: {105: 'cyl', 106: 'c1', 107: 'c2'}
        
        # Check if these nodes haven't already been collapsed by a previous match
        nodes_in_match = list(match.keys())
        if not all(n in brep_graph for n in nodes_in_match):
            continue
            
        # Create the new Macro Node
        new_macro_id = f"MACRO_CYL_{macro_id_counter}"
        brep_graph.add_node(new_macro_id, entity_type="<CYLINDER_PRIMITIVE>")
        macro_id_counter += 1
        
        # 4. Rewire external connections
        # Find all edges coming IN or OUT of the matched nodes from the REST of the graph
        for old_node in nodes_in_match:
            # Rewire incoming edges (e.g., an ADVANCED_FACE pointing to the CYLINDRICAL_SURFACE)
            for predecessor in list(brep_graph.predecessors(old_node)):
                if predecessor not in nodes_in_match:
                    brep_graph.add_edge(predecessor, new_macro_id)
            
            # Rewire outgoing edges (e.g., the CIRCLE pointing to a VERTEX_POINT)
            for successor in list(brep_graph.successors(old_node)):
                if successor not in nodes_in_match:
                    brep_graph.add_edge(new_macro_id, successor)
                    
        # 5. Delete the old verbose nodes
        brep_graph.remove_nodes_from(nodes_in_match)
        
    print(f"Compressed {macro_id_counter - 1} <CYLINDER_PRIMITIVE> macros.")
    return brep_graph

# ==========================================
# Example Usage 
# ==========================================
if __name__ == "__main__":
    # Mocking up a raw B-Rep graph parsed from a STEP file
    G = nx.DiGraph()
    G.add_node(1, entity_type="ADVANCED_FACE")
    G.add_node(2, entity_type="CYLINDRICAL_SURFACE") # The surface
    G.add_node(3, entity_type="CIRCLE")              # Top edge
    G.add_node(4, entity_type="CIRCLE")              # Bottom edge
    G.add_node(5, entity_type="VERTEX_POINT")
    
    G.add_edge(1, 2) # Face connects to surface
    G.add_edge(2, 3) # Surface bounded by circle 1
    G.add_edge(2, 4) # Surface bounded by circle 2
    G.add_edge(3, 5) # Circle 1 connects to a vertex
    
    print(f"Nodes before compression: {len(G.nodes)}")
    
    # Run the compression algorithm
    G_compressed = compress_cylinder_macros(G)
    
    print(f"Nodes after compression: {len(G_compressed.nodes)}")
    print(f"Current Graph Nodes: {G_compressed.nodes(data=True)}")

In [ ]:
import networkx as nx
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopAbs import TopAbs_FACE, TopAbs_EDGE
from OCC.Core.TopoDS import topods
from OCC.Core.BRepAdaptor import BRepAdaptor_Surface, BRepAdaptor_Curve
from OCC.Core.GeomAbs import (
    GeomAbs_Plane, GeomAbs_Cylinder, GeomAbs_Cone, GeomAbs_Sphere, 
    GeomAbs_Torus, GeomAbs_BSplineSurface, GeomAbs_Line, GeomAbs_Circle, 
    GeomAbs_Ellipse, GeomAbs_BSplineCurve
)

def get_surface_type_string(geom_type):
    """Maps OpenCASCADE surface enumerations to your STEP tokens."""
    mapping = {
        GeomAbs_Plane: "PLANE",
        GeomAbs_Cylinder: "CYLINDRICAL_SURFACE",
        GeomAbs_Cone: "CONICAL_SURFACE",
        GeomAbs_Sphere: "SPHERICAL_SURFACE",
        GeomAbs_Torus: "TOROIDAL_SURFACE",
        GeomAbs_BSplineSurface: "B_SPLINE_SURFACE_WITH_KNOTS"
    }
    return mapping.get(geom_type, "ADVANCED_FACE")

def get_curve_type_string(geom_type):
    """Maps OpenCASCADE curve enumerations to your STEP tokens."""
    mapping = {
        GeomAbs_Line: "LINE",
        GeomAbs_Circle: "CIRCLE",
        GeomAbs_Ellipse: "ELLIPSE",
        GeomAbs_BSplineCurve: "B_SPLINE_CURVE_WITH_KNOTS"
    }
    return mapping.get(geom_type, "EDGE_CURVE")

def step_to_networkx(filepath):
    """Reads a STEP file and outputs a raw topological nx.DiGraph."""
    reader = STEPControl_Reader()
    status = reader.ReadFile(filepath)
    
    if status != 1:
        raise Exception("Error: Could not read STEP file.")
        
    reader.TransferRoots()
    shape = reader.OneShape()
    
    brep_graph = nx.DiGraph()
    node_counter = 1
    
    # Dictionaries to keep track of shared edges
    edge_map = {} 
    
    # 1. Traverse all Faces in the STEP file
    face_explorer = TopExp_Explorer(shape, TopAbs_FACE)
    while face_explorer.More():
        face = topods.Face(face_explorer.Current())
        
        # Analyze the underlying mathematical surface
        surf_adaptor = BRepAdaptor_Surface(face)
        surf_type = get_surface_type_string(surf_adaptor.GetType())
        
        face_id = f"F_{node_counter}"
        brep_graph.add_node(face_id, entity_type=surf_type)
        node_counter += 1
        
        # 2. Traverse all Edges bounding this specific Face
        edge_explorer = TopExp_Explorer(face, TopAbs_EDGE)
        while edge_explorer.More():
            edge = topods.Edge(edge_explorer.Current())
            
            # Edges are often shared between faces. We hash them to avoid duplicates.
            edge_hash = edge.HashCode(2147483647) 
            
            if edge_hash not in edge_map:
                # Analyze the underlying mathematical curve
                curve_adaptor = BRepAdaptor_Curve(edge)
                curve_type = get_curve_type_string(curve_adaptor.GetType())
                
                edge_id = f"E_{node_counter}"
                brep_graph.add_node(edge_id, entity_type=curve_type)
                edge_map[edge_hash] = edge_id
                node_counter += 1
            else:
                edge_id = edge_map[edge_hash]
            
            # Create the topological link: Face is bounded by Edge
            brep_graph.add_edge(face_id, edge_id)
            
            edge_explorer.Next()
        face_explorer.Next()
        
    print(f"Successfully extracted {len(brep_graph.nodes)} nodes and {len(brep_graph.edges)} edges.")
    return brep_graph

# Example Usage
# graph = step_to_networkx("your_cad_model.step")
# print(graph.nodes(data=True))

# Dataset Preprocessing

In [ ]:
import google.generativeai as genai
import json
import time

# Configure the API key
# Get this from Google AI Studio: https://aistudio.google.com/

genai.configure(api_key=API_KEY)

# Initialize the model (using Flash for high-throughput data generation)
model = genai.GenerativeModel('gemini-2.5-flash')

def generate_cad_annotation(raw_geometry_text):
    prompt = f"""
    You are an expert mechanical engineer. Analyze the following raw CAD B-Rep geometry data and generate a hierarchical annotation strictly following these 3 levels.
    
    Level 1 (L1): Functional Category (What is the object conceptually? e.g., 'Hexagonal Bolt')
    Level 2 (L2): Topological Structure (How are the shapes arranged? e.g., 'A cylindrical shaft joined to a hexagonal driving head.')
    Level 3 (L3): Parametric Specification (What are the exact engineering dimensions? e.g., 'M8x1.25 standard hex bolt, shaft length 20mm, head height 5.3mm, spanner size 13mm.')

    Return ONLY the raw text in this exact format, with no markdown formatting or extra dialogue:
    [L1] <text>
    [L2] <text>
    [L3] <text>

    Raw Geometry Data:
    {raw_geometry_text}
    """
    
    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        print(f"API Error: {e}")
        return None

# Example Mock Data: What your Python OCC / STEP extraction script outputs
extracted_step_data = [
    {
        "file_id": "step_001.step",
        "raw_geometry": "1x Cylinder (Length: 20mm, Diameter: 8mm, Threaded: Yes, Pitch: 1.25mm), 1x Hexagonal Prism attached to top (Width across flats: 13mm, Height: 5.3mm)"
    },
    {
        "file_id": "step_002.step",
        "raw_geometry": "1x Rectangular Block (100mm x 50mm x 10mm), 4x Cylindrical Holes (Diameter: 5mm, Through-hole, located at corners 10mm from edges)"
    }
]

output_file = "cad_hierarchical_dataset.jsonl"

print(f"Starting generation for {len(extracted_step_data)} files...")

with open(output_file, 'w') as f:
    for item in extracted_step_data:
        print(f"Processing {item['file_id']}...")
        
        # Call the API
        annotation_text = generate_cad_annotation(item['raw_geometry'])
        
        if annotation_text:
            # We store it in JSONL for safe storage
            # The PyTorch Dataset will flatten this string later
            dataset_entry = {
                "file_id": item['file_id'],
                "raw_geometry_input": item['raw_geometry'],
                "hierarchical_target": annotation_text
            }
            
            # Write as a single JSON line
            f.write(json.dumps(dataset_entry) + "\n")
            
        # Add a slight delay to respect API rate limits
        time.sleep(2) 

print(f"Dataset successfully saved to {output_file}")

KeyboardInterrupt: 

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CADGeometricDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_length=512):
        """
        data_list: A list of dictionaries, where each dict contains:
        - 'text_sequence': String of the canonical DFS sequence + target description.
        - 'geometric_vectors': A list or numpy array of 128-dim vectors aligned with the geometric tokens.
        """
        self.data_list = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        
        # Stream A: Tokenize the sequence
        encoded = self.tokenizer(
            item['text_sequence'],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        
        input_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)
        
        # Stream B: Prepare the geometric tensor
        # In a real scenario, you map your pre-calculated 128-dim vectors to the exact index of the geometric tokens.
        # For non-geometric tokens (like regular English words), the vector should be zeros.
        # Here we mock it up as a zero tensor, and insert the actual vectors at the right indices.
        geom_tensor = torch.zeros((self.max_length, 128))
        
        # (Your logic here to populate geom_tensor based on the indices of <FACE>, <CYLINDER>, etc.)
        # Example: geom_tensor[5] = torch.tensor(item['geometric_vectors'][0])
        
        # Create labels for causal language modeling (shift right is handled by the model internally)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100 # Ignore padding in loss calculation
        
        return {
            "input_ids": input_ids,
            "geometric_vectors": geom_tensor,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [ ]:
class DualStreamCADModel(nn.Module):
    def __init__(self, model_id, tokenizer_len):
        super().__init__()
        
        # 1. Load the frozen base LLM (using bfloat16 for memory efficiency)
        self.base_model = AutoModelForCausalLM.from_pretrained(
            model_id, 
            torch_dtype=torch.bfloat16,
            device_map="auto" 
        )
        
        # 2. Resize the embedding matrix to accommodate <FACE>, etc.
        self.base_model.resize_token_embeddings(tokenizer_len)
        
        # 3. Apply LoRA (Low-Rank Adaptation)
        peft_config = LoraConfig(
            r=16, 
            lora_alpha=32,
            target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Target attention layers
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        self.base_model = get_peft_model(self.base_model, peft_config)
        
        # 4. Define W_p: The Linear Projection Matrix for Stream B (128 -> d_model)
        # Llama 3 8B hidden size is usually 4096. We pull it dynamically.
        d_model = self.base_model.config.hidden_size 
        self.W_p = nn.Linear(128, d_model, dtype=torch.bfloat16).to(self.base_model.device)
        
        # Ensure W_p and the new token embeddings require gradients!
        self.W_p.weight.requires_grad = True
        self.base_model.get_input_embeddings().weight.requires_grad = True

    def forward(self, input_ids, geometric_vectors, attention_mask=None, labels=None):
        # E_semantic(t): Get standard word embeddings from Stream A
        # Shape: (batch_size, sequence_length, d_model)
        E_semantic = self.base_model.get_input_embeddings()(input_ids)
        
        # V_complex * W_p: Project Stream B into the latent space
        # Shape: (batch_size, sequence_length, d_model)
        V_projected = self.W_p(geometric_vectors.to(dtype=torch.bfloat16, device=self.base_model.device))
        
        # Equation 3.4: The Fusion
        E_input = E_semantic + V_projected
        
        # Pass the fused embeddings directly into the LLM instead of input_ids
        outputs = self.base_model(
            inputs_embeds=E_input, 
            attention_mask=attention_mask,
            labels=labels
        )
        
        return outputs

# Initialize the model
custom_model = DualStreamCADModel(model_id, len(tokenizer))
print("Model initialized and adapted successfully.")

In [ ]:
import torch
import time

def benchmark_inference(model, input_embeddings, max_new_tokens=50):
    # Ensure model is in evaluation mode
    model.eval()
    
    # Create CUDA events for precise timing
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    # 1. Warm-up run (Crucial!)
    # The first run on a GPU is always slow due to memory initialization. 
    # We run a dummy pass to ensure the GPU is warm.
    with torch.no_grad():
        _ = model.generate(inputs_embeds=input_embeddings, max_new_tokens=10)
    
    # 2. The Actual Benchmark
    torch.cuda.synchronize() # Wait for warm-up to completely finish
    start_event.record()     # Start the stopwatch
    
    with torch.no_grad():
        outputs = model.generate(
            inputs_embeds=input_embeddings, 
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
        
    end_event.record()       # Stop the stopwatch
    torch.cuda.synchronize() # Wait for GPU to finish generating
    
    # 3. Calculate Metrics
    # elapsed_time returns milliseconds, convert to seconds
    latency_sec = start_event.elapsed_time(end_event) / 1000.0 
    
    # How many tokens did the model actually generate?
    # (outputs shape is usually [batch_size, sequence_length])
    generated_tokens = outputs.shape[1] 
    
    throughput = generated_tokens / latency_sec
    
    print(f"--- Benchmark Results ---")
    print(f"Tokens Generated: {generated_tokens}")
    print(f"Total Latency:    {latency_sec:.4f} seconds")
    print(f"Throughput:       {throughput:.2f} tokens/second")
    
    return outputs